# Bumpy and smooth recording comparison

This notebook compares one **bumpy** and one **smooth** accelerometer recording from Lars. Using the same participant avoids mixing the surface comparison with a rider/phone-placement difference.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'Bumpy': '#D97706', 'Smooth': '#2563EB'}
AXIS_COLORS = {'x': '#2563EB', 'y': '#D97706', 'z': '#4D7C0F'}

## Load two recordings

In [2]:
DATA_ROOT = Path('data/recordings/raw')
RECORDINGS = {
    'Bumpy': DATA_ROOT / 'bumpy/lars/cb_2_bumpy_lars/Accelerometer.csv',
    'Smooth': DATA_ROOT / 'smooth/lars/SSC-smooth-test1-lars/Accelerometer.csv',
}

def load_accelerometer(path):
    data = pd.read_csv(path)
    required = {'seconds_elapsed', 'x', 'y', 'z'}
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(f'{path} is missing columns: {sorted(missing)}')
    data = data.sort_values('seconds_elapsed').copy()
    data['seconds_elapsed'] -= data['seconds_elapsed'].iloc[0]
    data['magnitude'] = np.sqrt((data[['x', 'y', 'z']] ** 2).sum(axis=1))
    return data

recordings = {label: load_accelerometer(path) for label, path in RECORDINGS.items()}
pd.DataFrame({
    label: {'samples': len(data), 'duration_s': data['seconds_elapsed'].iloc[-1]}
    for label, data in recordings.items()
}).T.round(2)

FileNotFoundError: [Errno 2] No such file or directory: 'data/recordings/raw/bumpy/lars/cb_2_bumpy_lars/Accelerometer.csv'

## Acceleration magnitude

Both panels use the same y-axis. Magnitude combines the x, y, and z components, so it is less sensitive to how the phone is rotated.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True, constrained_layout=True)

for axis, (label, data) in zip(axes, recordings.items()):
    axis.plot(data['seconds_elapsed'], data['magnitude'], color=COLORS[label], linewidth=0.9)
    axis.set_title(f'{label}: acceleration magnitude', loc='left', fontweight='bold')
    axis.set_ylabel('Acceleration (m/s²)')
    axis.grid(axis='x', alpha=0.2)
    axis.grid(axis='y', alpha=0.35)

axes[-1].set_xlabel('Elapsed time (s)')
fig.suptitle('One bumpy and one smooth bicycle recording', fontsize=14, fontweight='bold')
plt.show()

## Individual sensor axes

These panels keep the same axis colors and y-scale, which makes spikes and directional differences directly comparable.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, sharey=True, constrained_layout=True)

for axis, (label, data) in zip(axes, recordings.items()):
    for component in ['x', 'y', 'z']:
        axis.plot(
            data['seconds_elapsed'], data[component],
            label=component.upper(), color=AXIS_COLORS[component], linewidth=0.75, alpha=0.9,
        )
    axis.axhline(0, color='#374151', linewidth=0.7)
    axis.set_title(label, loc='left', fontweight='bold')
    axis.set_ylabel('Acceleration (m/s²)')
    axis.legend(ncols=3, frameon=False, loc='upper right')
    axis.grid(axis='x', alpha=0.2)
    axis.grid(axis='y', alpha=0.35)

axes[-1].set_xlabel('Elapsed time (s)')
fig.suptitle('Accelerometer components by surface', fontsize=14, fontweight='bold')
plt.show()

## Interpretation

Look for larger and more frequent spikes in the bumpy recording. This is a visual check of two example rides, not evidence that every bumpy ride is separable from every smooth ride.